# Flood Mapping and Vulnerability Assessment in River and Lagoon Environments

*by Daniele Petrini & Federico Urbini*

## Necessary Imports

In [1]:
import ee
import pandas as pd
from pathlib import Path
import importlib
from IPython.display import display

from utils import config as cfg
from utils import utils_setup as us
from utils import utils_SAR as su
from utils import utils_optical as ou
from utils import utils_final as uf

importlib.reload(cfg)
importlib.reload(us)
importlib.reload(uf)

<module 'utils.utils_final' from '/home/dani/Desktop/eoda/floods-SAR/utils/utils_final.py'>

## GEE Authentication

In [2]:
ee.Authenticate(auth_mode="localhost")
ee.Initialize(project="floods-2024")
print("Earth Engine initialized correctly.")

Earth Engine initialized correctly.


## Folders

In [3]:
Path("outputs").mkdir(exist_ok=True)
Path("outputs/tables").mkdir(parents=True, exist_ok=True)
Path("outputs/figures").mkdir(parents=True, exist_ok=True)
Path("outputs/maps").mkdir(parents=True, exist_ok=True)
Path("outputs/final").mkdir(parents=True, exist_ok=True)

## Global Flags

In [4]:
MONTH = "sept"  # Options: "sept", "oct"
HAZARD_SOURCE = "fusion"  # Options: "fusion", "sar"

MAKE_MAPS = True
MAKE_SETUP_MAPS = True
MAKE_SAR_MAP = True
MAKE_OPTICAL_MAP = True
MAKE_FINAL_MAP = True

OUTPUT_TABLE_DIR = Path("outputs") / "tables"
OUTPUT_MAP_DIR = Path("outputs") / "maps"
FINAL_MAP_DIR = Path("outputs") / "final"

INCLUDE_SEASONAL_REFERENCE = True
SHOW_RIVERS_BY_DEFAULT = False
SHOW_S2_PREVIEWS_BY_DEFAULT = False
SHOW_WATER_ON_EVENT_MAP = False

# Options: "JRC", "PERMANENT_WATER", "WORLDCOVER", "DEM", "SLOPE", "NONE"
AUX_DEFAULT_LAYER = "JRC"

## Config Params

In [5]:
aoi_emilia_romagna = cfg.get_project_config()["aoi_emilia_romagna"]
aoi_romagna = cfg.get_project_config()["aoi_romagna"]
aoi_bologna = cfg.get_project_config()["aoi_bologna"]
places = cfg.get_project_config()["places"]
dates = cfg.get_project_config()["dates"]
events = cfg.get_project_config()["events"]

In [6]:
# Create transparent AOI outlines and styled city points.
aoi_styles = {
    "Broad AOI": ee.FeatureCollection([ee.Feature(aoi_emilia_romagna)]).style(color="yellow", fillColor="00000000", width=2),
    "AOI Romagna": ee.FeatureCollection([ee.Feature(aoi_romagna)]).style(color="red", fillColor="00000000", width=2),
    "AOI Bologna/Pianoro": ee.FeatureCollection([ee.Feature(aoi_bologna)]).style(color="blue", fillColor="00000000", width=2),
}

event_aoi_styles = {
    "sept": aoi_styles["AOI Romagna"],
    "oct": aoi_styles["AOI Bologna/Pianoro"],
}

places_styles = {
    "emilia": places.filterBounds(aoi_emilia_romagna).style(color="000000", pointSize=5,pointShape="circle"),
    "sept": places.filterBounds(aoi_romagna).style(color="000000", pointSize=5, pointShape="circle"),
    "oct": places.filterBounds(aoi_bologna).style(color="000000", pointSize=5, pointShape="circle"),
}

## Datasets and Visualization

In [7]:
# Load datasets
data = us.load_core_datasets(
    us=us,
    events=events,
    dates=dates,
    aoi_emilia_romagna=aoi_emilia_romagna,
    aoi_romagna=aoi_romagna,
    include_seasonal_reference=INCLUDE_SEASONAL_REFERENCE,
)

s1 = data["s1"]
s1_orbits = data["s1_orbits"]
s2 = data["s2"]

auxiliary = data["auxiliary"]
water_occurrence = data["water_occurrence"]
permanent_water = data["permanent_water"]
rivers_image = data["rivers_image"]
vis = data["vis"]

s1_seasonal_ref = data["s1_seasonal_ref"]
s2_seasonal_ref_romagna = data["s2_seasonal_ref_romagna"]

print("Datasets loaded.")

Datasets loaded.


In [8]:
# Build preview layers and availability table
preview_data = us.build_preview_layers_and_availability(
    us=us,
    events=events,
    s1=s1,
    s1_orbits=s1_orbits,
    s2=s2,
    include_seasonal_reference=INCLUDE_SEASONAL_REFERENCE,
    s1_seasonal_ref=s1_seasonal_ref,
    s2_seasonal_ref_romagna=s2_seasonal_ref_romagna,
    aoi_romagna=aoi_romagna,
    output_path="outputs/tables/00_dataset_availability.csv",
)

s1_vv = preview_data["s1_vv"]
s2_rgb = preview_data["s2_rgb"]
seasonal_layers = preview_data["seasonal_layers"]
df_availability = preview_data["df_availability"]

display(df_availability)

,Dataset window,Number of images
0,S1 September Romagna pre-event ALL,14
1,S1 September Romagna post-event ALL,3
2,S1 September Romagna recovery ALL,10
3,S1 September Romagna pre-event ASC,10
4,S1 September Romagna post-event ASC,2
5,S1 September Romagna pre-event DESC,4
6,S1 September Romagna post-event DESC,1
7,S2 September Romagna pre-event raw,52
8,S2 September Romagna post-event raw,16
9,S2 September Romagna recovery raw,25


In [9]:
# Map 1: September flood event, Romagna
Map_sept = us.make_setup_event_map(
    event_key="sept",
    events=events,
    aoi_style=event_aoi_styles["sept"],
    places_style=places_styles["sept"],
    rivers_image=rivers_image,
    s1_vv=s1_vv,
    s2_rgb=s2_rgb,
    water_occurrence=water_occurrence,
    permanent_water=permanent_water,
    vis=vis,
    include_seasonal_reference=INCLUDE_SEASONAL_REFERENCE,
    show_rivers_by_default=SHOW_RIVERS_BY_DEFAULT,
    show_s2_previews_by_default=SHOW_S2_PREVIEWS_BY_DEFAULT,
    show_water_on_event_map=SHOW_WATER_ON_EVENT_MAP,
    seasonal_layers=seasonal_layers,
)

Map_sept

Map(center=[44.35, 11.9], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…

In [10]:
# Map 2: October flood event, Bologna/Pianoro
Map_oct = us.make_setup_event_map(
    event_key="oct",
    events=events,
    aoi_style=event_aoi_styles["oct"],
    places_style=places_styles["oct"],
    rivers_image=rivers_image,
    s1_vv=s1_vv,
    s2_rgb=s2_rgb,
    water_occurrence=water_occurrence,
    permanent_water=permanent_water,
    vis=vis,
    include_seasonal_reference=INCLUDE_SEASONAL_REFERENCE,
    show_rivers_by_default=SHOW_RIVERS_BY_DEFAULT,
    show_s2_previews_by_default=SHOW_S2_PREVIEWS_BY_DEFAULT,
    show_water_on_event_map=SHOW_WATER_ON_EVENT_MAP,
    seasonal_layers=seasonal_layers,
)

Map_oct

Map(center=[44.43, 11.35], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright'…

In [11]:
# Map 3: Auxiliary layers for interpretation
Map_aux = us.make_auxiliary_map(
    center=[44.35, 11.8],
    zoom=8,
    aoi=aoi_emilia_romagna,
    aoi_styles=aoi_styles,
    places_style=places_styles["emilia"],
    rivers_image=rivers_image,
    auxiliary=auxiliary,
    vis=vis,
    default_layer=AUX_DEFAULT_LAYER.lower(),
)

Map_aux

Map(center=[44.35, 11.8], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…

## SAR Data Implementation

In [12]:
# Load auxiliary datasets for SAR
auxiliary = us.load_auxiliary_datasets(aoi_emilia_romagna)

permanent_water = auxiliary["permanent_water"]
slope = auxiliary["slope"]
rivers_image = auxiliary["rivers_image"]

vis = su.get_sar_vis_params()

print("Configuration and auxiliary datasets loaded.")

Configuration and auxiliary datasets loaded.


In [13]:
# Run SAR flood mapping for the selected event
results = su.run_sar_event(
    event_key=MONTH,
    event=events[MONTH],
    dates=dates,
    permanent_water=permanent_water,
    slope=slope,
    rivers_image=rivers_image,
    places=places,
    vis=vis,
    output_dir=OUTPUT_TABLE_DIR,
    make_map=MAKE_MAPS,
    **cfg.SAR_PARAMS[MONTH],
)

display(results["df_candidates"])
display(results["df_metadata"])
display(results["df_area"])

print("Selected Sentinel-1 geometry:")
print(results["selected"])

print("Otsu threshold:", results["otsu_threshold"])
print("Otsu physically valid:", results["otsu_valid"])

,orbit_pass,relative_orbit,pre_count,post_count,balanced_count,total_count
0,ASCENDING,15,2,1,1,3
1,ASCENDING,44,2,1,1,3
2,DESCENDING,168,2,1,1,3


,date,label,mean_incidence_angle,orbit_pass,polarizations,relative_orbit,resolution_meters,system_index
0,2024-09-18 05:27,sept_post,31.164600,DESCENDING,VV/VH,168,10,S1A_IW_GRDH_1SDV_20240918T052757_20240918T0528...
1,2024-09-19 17:14,sept_post,45.864226,ASCENDING,VV/VH,15,10,S1A_IW_GRDH_1SDV_20240919T171453_20240919T1715...
2,2024-09-21 16:58,sept_post,30.670397,ASCENDING,VV/VH,44,10,S1A_IW_GRDH_1SDV_20240921T165840_20240921T1659...
3,2024-08-20 05:19,sept_pre,39.518365,DESCENDING,VV/VH,95,10,S1A_IW_GRDH_1SDV_20240820T051947_20240820T0520...
4,2024-08-21 17:06,sept_pre,38.782118,ASCENDING,VV/VH,117,10,S1A_IW_GRDH_1SDV_20240821T170631_20240821T1706...
5,2024-08-21 17:06,sept_pre,38.543054,ASCENDING,VV/VH,117,10,S1A_IW_GRDH_1SDV_20240821T170656_20240821T1707...
6,2024-08-25 05:27,sept_pre,31.204167,DESCENDING,VV/VH,168,10,S1A_IW_GRDH_1SDV_20240825T052756_20240825T0528...
7,2024-08-26 17:14,sept_pre,45.864466,ASCENDING,VV/VH,15,10,S1A_IW_GRDH_1SDV_20240826T171452_20240826T1715...
8,2024-08-28 16:58,sept_pre,30.688373,ASCENDING,VV/VH,44,10,S1A_IW_GRDH_1SDV_20240828T165839_20240828T1659...
9,2024-09-02 17:06,sept_pre,38.781936,ASCENDING,VV/VH,117,10,S1A_IW_GRDH_1SDV_20240902T170631_20240902T1706...


,event,method,used_in_confidence,area_km2
0,sept,delta_VV_lt_-2.5_dB,True,38.295058
1,sept,ratio_lt_0.7,True,72.125182
2,sept,VV_post_lt_-17_dB,True,26.473151
3,sept,otsu_delta_VV_lt_-1.484_dB,False,75.023261
4,sept,sar_confidence_medium_high,True,44.471016
5,sept,sar_confidence_high,True,11.971913


Selected Sentinel-1 geometry:
{'selection_mode': 'manual_override', 'orbit_pass': 'ASCENDING', 'relative_orbit': 44, 'pre_image_count': 2, 'post_image_count': 1}
Otsu threshold: -1.4838502626889178
Otsu physically valid: True


In [14]:
# Display and save SAR map as an interactive HTML file
display(results["map"])
map_path = OUTPUT_MAP_DIR / f"{MONTH}_sar_map.html"
results["map"].to_html(str(map_path))
print(f"Map saved to: {map_path}")

Map(center=[44.35, 11.9], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…

Map saved to: outputs/maps/sept_sar_map.html


In [15]:
# Save SAR area summary
df_sar_area_summary = results["df_area"].copy()

display(df_sar_area_summary)

df_sar_area_summary.to_csv(
    OUTPUT_TABLE_DIR / f"{MONTH}_sar_area_summary.csv",
    index=False,
)

,event,method,used_in_confidence,area_km2
0,sept,delta_VV_lt_-2.5_dB,True,38.295058
1,sept,ratio_lt_0.7,True,72.125182
2,sept,VV_post_lt_-17_dB,True,26.473151
3,sept,otsu_delta_VV_lt_-1.484_dB,False,75.023261
4,sept,sar_confidence_medium_high,True,44.471016
5,sept,sar_confidence_high,True,11.971913


In [16]:
# Save selected Sentinel-1 geometry summary
df_selected_s1 = pd.DataFrame([
    {
        "event": MONTH,
        **results["selected"],
        "otsu_threshold_delta_vv": results["otsu_threshold"],
        "otsu_valid": results["otsu_valid"],
    }
])

display(df_selected_s1)

df_selected_s1.to_csv(
    OUTPUT_TABLE_DIR / f"{MONTH}_selected_s1_geometry_summary.csv",
    index=False,
)

,event,selection_mode,orbit_pass,relative_orbit,pre_image_count,post_image_count,otsu_threshold_delta_vv,otsu_valid
0,sept,manual_override,ASCENDING,44,2,1,-1.48385,True


## Optical Data Implementation

In [17]:
# Run Sentinel-2 optical processing and SAR-optical fusion for the selected event
opt_vis = ou.get_optical_vis_params()

optical_results = ou.run_optical_event(
    event_key=MONTH,
    event=events[MONTH],
    dates=dates,
    sar_confidence=results["sar_confidence"],
    permanent_water=permanent_water,
    places=places,
    output_dir=OUTPUT_TABLE_DIR,
    make_map=MAKE_OPTICAL_MAP,
    vis=opt_vis,
    **cfg.OPTICAL_PARAMS[MONTH],
)

display(optical_results["df_s2_metadata"])
display(optical_results["df_valid_summary"])
display(optical_results["df_agreement"])

,cloudy_pixel_percentage,date,label,mgrs_tile,system_index
0,92.319429,2024-09-19 10:18,sept_post,32TPP,20240919T101019_20240919T101702_T32TPP
1,99.147075,2024-09-19 10:18,sept_post,32TPQ,20240919T101019_20240919T101702_T32TPQ
2,99.967992,2024-09-19 10:18,sept_post,32TQP,20240919T101019_20240919T101702_T32TQP
3,99.999452,2024-09-19 10:18,sept_post,32TQQ,20240919T101019_20240919T101702_T32TQQ
4,99.997205,2024-09-19 10:18,sept_post,33TUJ,20240919T101019_20240919T101702_T33TUJ
...,...,...,...,...,...
63,93.543178,2024-09-14 10:18,sept_pre,32TPQ,20240914T100631_20240914T100718_T32TPQ
64,42.533979,2024-09-14 10:18,sept_pre,32TQP,20240914T100631_20240914T100718_T32TQP
65,89.395469,2024-09-14 10:18,sept_pre,32TQQ,20240914T100631_20240914T100718_T32TQQ
66,61.121106,2024-09-14 10:18,sept_pre,33TUJ,20240914T100631_20240914T100718_T33TUJ


,event,window,valid_pixel_percentage_aoi
0,sept,pre,99.882410
1,sept,post,34.142835


,sar_area_km2_valid_optical,optical_area_km2,intersection_km2,union_km2,iou,dice,event,optical_index,optical_threshold
0,32.100245,50.834384,9.800303,69.749637,0.140507,0.236338,sept,MNDWI,0.0


In [18]:
# Display optical and SAR-optical fusion map
display(optical_results["map"])
map_path = OUTPUT_MAP_DIR / f"{MONTH}_optical_map.html"
optical_results["map"].to_html(str(map_path))
print(f"Map saved to: {map_path}")

Map(center=[44.35, 11.9], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…

Map saved to: outputs/maps/sept_optical_map.html


In [19]:
# Save SAR-optical agreement summary
df_sar_optical_summary = optical_results["df_agreement"].copy()

display(df_sar_optical_summary)

df_sar_optical_summary.to_csv(
    OUTPUT_TABLE_DIR / f"{MONTH}_sar_optical_agreement_summary.csv",
    index=False,
)

,sar_area_km2_valid_optical,optical_area_km2,intersection_km2,union_km2,iou,dice,event,optical_index,optical_threshold
0,32.100245,50.834384,9.800303,69.749637,0.140507,0.236338,sept,MNDWI,0.0


## Risk Analysis

In [20]:
# Load auxiliary datasets for risk analysis
auxiliary = us.load_auxiliary_datasets(aoi_emilia_romagna)
water_occurrence = auxiliary["water_occurrence"]
worldcover = auxiliary["worldcover"]
dem = auxiliary["dem"]

risk_vis = uf.get_risk_vis_params()

print("Configuration and auxiliary datasets loaded.")

Configuration and auxiliary datasets loaded.


In [21]:
# Select hazard input for the final risk analysis
if HAZARD_SOURCE == "sar":
    if "results" not in globals():
        raise RuntimeError(
            "SAR results are missing. Run the SAR processing cell before the final risk analysis."
        )

    hazard_input = results["sar_confidence"]

elif HAZARD_SOURCE == "fusion":
    if "optical_results" not in globals():
        raise RuntimeError(
            "Optical/fusion results are missing. Run the optical/fusion processing cell before the final risk analysis."
        )

    hazard_input = optical_results["fusion_map"]

else:
    raise ValueError("HAZARD_SOURCE must be either 'fusion' or 'sar'.")

print("Selected hazard source:", HAZARD_SOURCE)

Selected hazard source: fusion


In [22]:
# Run vulnerability, exposure, and flood risk analysis
risk_results = uf.run_final_risk_event(
    event_key=MONTH,
    event=events[MONTH],
    hazard_input=hazard_input,
    hazard_source=HAZARD_SOURCE,
    dem=dem,
    slope=slope,
    worldcover=worldcover,
    water_occurrence=water_occurrence,
    places=places,
    output_dir=OUTPUT_TABLE_DIR,
    make_map=MAKE_FINAL_MAP,
    vis=risk_vis,
    **cfg.RISK_PARAMS[MONTH],
)

print("Low elevation threshold:", risk_results["low_elevation_threshold"].getInfo())

display(risk_results["df_exposure"])
display(risk_results["df_vulnerability_area"])
display(risk_results["df_risk_area"])
display(risk_results["df_roi"])

Low elevation threshold: 2.1312052611295944


,event,worldcover_class,worldcover_label,area_km2
0,sept,10,Tree cover,3.447080
1,sept,20,Shrubland,0.303954
2,sept,30,Grassland,6.529242
3,sept,40,Cropland,11.919267
4,sept,50,Built-up,4.902419
5,sept,60,Bare / sparse vegetation,2.501136
6,sept,80,Permanent water bodies,13.300923
7,sept,90,Herbaceous wetland,1.566994


,event,class,label,area_km2
0,sept,0,No / very low vulnerability,1633.742652
1,sept,1,Low vulnerability,1887.765272
2,sept,2,Medium vulnerability,4607.396036
3,sept,3,High vulnerability,369.210276


,event,class,label,area_km2
0,sept,0,No / very low risk,8371.943785
1,sept,1,Low risk,88.970893
2,sept,2,Medium risk,35.384607
3,sept,3,High risk,1.810295


,event,name,buffer_m,flood_area_km2,high_risk_area_km2,builtup_flooded_km2,cropland_flooded_km2
0,sept,Faenza,3000,0,0,0,0
1,sept,Castel Bolognese,3000,0,0,0,0
2,sept,Lugo,3000,0,0,0,0
3,sept,Bagnacavallo,3000,0,0,0,0
4,sept,Ravenna,3000,0,0,0,0
5,sept,Bologna,3000,0,0,0,0
6,sept,Pianoro,3000,0,0,0,0


In [23]:
# Display and save final risk map as an interactive HTML file
display(risk_results["map"])

map_path = FINAL_MAP_DIR / f"{MONTH}_final_risk_map.html"
risk_results["map"].to_html(str(map_path))
print(f"Final risk map saved to: {map_path}")

Map(center=[44.35, 11.9], controls=(WidgetControl(options=['position', 'transparent_bg'], position='topright',…

Final risk map saved to: outputs/final/sept_final_risk_map.html


## Final Results

In [24]:
# Save compact final summary tables

risk_results["df_exposure"].to_csv(
    OUTPUT_TABLE_DIR / f"{MONTH}_final_exposure_summary.csv",
    index=False,
)

risk_results["df_risk_area"].to_csv(
    OUTPUT_TABLE_DIR / f"{MONTH}_final_risk_summary.csv",
    index=False,
)

risk_results["df_roi"].to_csv(
    OUTPUT_TABLE_DIR / f"{MONTH}_final_roi_summary.csv",
    index=False,
)

print("Final summary tables saved.")

Final summary tables saved.
